# Logistic Regression — Student Dropout Prediction

This notebook builds a multiclass Logistic Regression model to predict student outcomes (Graduate, Dropout, or Enrolled). It reuses the dataset and key engineered features identified in the group's EDA, then evaluates the model with accuracy, a classification report, a confusion matrix, and ROC/AUC curves.

## Step 1: Load Data and Prepare Features
We load the same dataset used in the EDA and recreate the "approval ratio" feature the team engineered (approved units ÷ enrolled units), since this notebook runs independently. We then separate the data into features (X) and the target (y), which has three classes: Graduate, Dropout, and Enrolled. We also check the class balance, because an imbalanced target affects how we interpret accuracy later.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load with semicolon separator (this dataset uses ; not ,)
df = pd.read_csv("data.csv", sep=';')

# Recreate the engineered feature from the EDA (approval ratio)
for sem in ['1st', '2nd']:
    enrolled = df[f'Curricular units {sem} sem (enrolled)']
    approved = df[f'Curricular units {sem} sem (approved)']
    df[f'Approval ratio {sem} sem'] = np.where(enrolled > 0, approved / enrolled, 0)

# Features (X) and target (y)
X = df.drop(columns=['Target'])
y = df['Target']

print("Shape:", X.shape)
print("Class balance:")
print(y.value_counts(normalize=True).round(3).to_string())

## Step 2: Train/Test Split and Feature Scaling
We split the data into training and test sets, using `stratify=y` so each set keeps the same proportion of the three outcomes. We then scale the features with StandardScaler, because Logistic Regression is sensitive to feature magnitude and this dataset mixes very different scales (grades from 0–20, ages, and binary 0/1 flags). The scaler is fit only on the training data to avoid data leakage from the test set.

In [0]:
# Split into train/test, stratified to keep class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (fit on train only, then apply to both)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Step 3: Train the Model and Evaluate
We train a multiclass Logistic Regression (it handles three classes automatically using a one-vs-rest scheme internally). We then evaluate it three ways: overall accuracy, a per-class classification report (precision, recall, F1 for each outcome), and a confusion matrix showing exactly where the model confuses one outcome for another. The per-class view matters most here, since correctly identifying *Dropout* students is the real goal of the project.

In [0]:
# Train multiclass logistic regression
model = LogisticRegression(max_iter=5000)
model.fit(X_train_scaled, y_train)

# Predict and evaluate
y_pred = model.predict(X_test_scaled)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=model.classes_, yticklabels=model.classes_)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

### Logistic Regression — Results & Interpretation

The multiclass Logistic Regression achieved **76.8% overall accuracy**. However, accuracy alone is misleading here because the classes are imbalanced (Graduate 50%, Dropout 32%, Enrolled 18%), so the per-class metrics tell the real story:

- **Graduate** is predicted best (recall 0.93) — it's the largest class with the clearest signal (strong grades, fees paid).
- **Dropout** is predicted well (recall 0.77) — most importantly for this project, the model catches roughly 3 out of 4 students who actually drop out, making it genuinely useful for early intervention.
- **Enrolled** is the weakest (recall 0.32) — this is expected: it's the smallest class and the most ambiguous outcome, since "still enrolled" students share features with both graduates and dropouts, so the model frequently misclassifies them as one of the other two.

**Key takeaway:** For the project's goal of identifying at-risk students early, the Dropout recall of 0.77 is the most relevant metric. The model would correctly flag the majority of at-risk students. The main weakness is distinguishing the middle "Enrolled" group, which is inherently the hardest to separate.

## Step 4: ROC Curves and AUC (One-vs-Rest)
Since this is a 3-class problem, we evaluate the model's ability to separate each outcome from the rest using a One-vs-Rest approach: one ROC curve per class (Graduate vs rest, Dropout vs rest, Enrolled vs rest). Each curve shows the tradeoff between catching true positives and raising false alarms across all thresholds, and the AUC summarizes each into a single score. This is threshold-independent, so it complements the accuracy and recall numbers above and is especially useful given the class imbalance. Higher AUC (closer to 1.0) means better separation.

In [0]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Binarize the 3-class target into one column per class (One-vs-Rest)
classes = model.classes_   # ['Dropout', 'Enrolled', 'Graduate']
y_test_bin = label_binarize(y_test, classes=classes)

# Get predicted probabilities for each class
y_score = model.predict_proba(X_test_scaled)

# Plot one ROC curve per class
plt.figure(figsize=(8, 6))
colors = ['#ff1d6c', '#fbbf24', '#2dd4bf']
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2,
             label=f'{cls} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves (One-vs-Rest) — Logistic Regression')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### ROC / AUC Interpretation

All three classes separate well from the rest, with every curve sitting far above the random-guess diagonal:
- **Graduate (AUC 0.92)** and **Dropout (AUC 0.91)** are both "outstanding," confirming the model distinguishes these outcomes very reliably.
- **Enrolled (AUC 0.80)** is the weakest but still "excellent."

A key insight: Enrolled's AUC (0.80) is much more encouraging than its earlier recall (0.32). This is because AUC measures the model's ability to *rank* students by probability across all thresholds, while recall reflects the final hard prediction at the default threshold. The model actually assigns meaningfully higher Enrolled-probabilities to true Enrolled students, but in a 3-class setup those probabilities are often still lower than the competing Graduate or Dropout scores, so Enrolled rarely "wins" the final prediction. This suggests the Enrolled performance could be improved by adjusting class weights or decision thresholds, rather than the model fundamentally failing to recognize the class.

For the project's goal of early intervention, the **Dropout AUC of 0.91** is especially strong: the model is highly capable of ranking at-risk students, which is exactly what an institution needs to flag students for support.